# Vehicle Counting with Object Tracking: 도로 주행 차량 차종별 집계 및 체류 시간 분석

이 노트북은 도로를 주행하는 차량 영상에서 **RF-DETR(`RFDETRSmall`)** 객체 검출 모델과 **ByteTrack** 다중 객체 추적(Multi-Object Tracking) 알고리즘을 결합하여, 지정된 기준선(Line Zone)을 통과하는 차량의 대수를 방향(IN / OUT) 및 차종별로 집계하는 실습 자료입니다.

### 왜 단순 검출(Detection)을 넘어 객체 추적(Object Tracking)이 필요한가?
1. **중복 카운팅 방지 (박스 누적 vs 실제 차량 수)**: 단순히 매 프레임마다 검출된 바운딩 박스의 개수만 세면, 동일한 차량이 화면에 3초(약 75프레임)만 머물러도 75대로 중복 집계됩니다. 반면 추적기(Tracker)로 각 차량에 고유한 **추적 ID(`#ID`)**를 부여하면, 화면 전체에서의 누적 박스 수(Raw Detections)가 수천 개에 달하더라도 **고유 차량 수(Unique IDs)**와 **기준선 통과 대수(Line Crossed)**를 정확하게 분리하여 집계할 수 있습니다.
2. **시간($\Delta t$)과 공간($\Delta s$)의 결합 (체류 시간 및 이동 궤적)**: 트래커는 각 차량이 처음 등장한 프레임부터 현재 프레임까지의 생애 주기(Lifecycle)를 관리합니다. 이를 통해 각 차량 라벨에 **화면 내 체류 시간(Dwell Time, 예: `2.4s`)**을 실시간으로 계산해 표시하고 이동 궤적(Trace)을 그릴 수 있으며, 이는 교통 정체 분석이나 불법 주정차 감지의 핵심 원리가 됩니다.

> **실행 전 준비 사항**
> 원활한 딥러닝 추론 및 영상 처리를 위해 상단 메뉴에서 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU(T4)**를 먼저 선택해 주세요.

### 제공되는 고속도로 예시 영상
주차된 차량이 많은 일반 시내 거리 대신, 차로를 따라 차량이 연속해서 주행하는 고속도로 장면 3가지를 예시로 제공합니다.

| 예시 이름 (`EXAMPLE`) | 파일명 | 장면 설명 |
| :--- | :--- | :--- |
| **`vehicles`** | `examples/vehicles.mp4` | Supervision 기본 제공 고속도로 영상 (가로 기준선 사용) |
| **`a40`** | `examples/a40.mp4` | 독일 보훔(Bochum) A40 아우토반 고가교에서 내려다본 양방향 도로 |
| **`a43`** | `examples/a43.mp4` | 독일 보훔(Bochum) A43 아우토반 고가교 양방향 도로 (가까운 차량이 크게 보임) |


## 1. 필수 패키지 설치 및 하드웨어(GPU) 환경 확인

차량 검출을 위한 `rfdetr`와 객체 추적(`ByteTrack`), 기준선 집계(`LineZone`), 시각화 어노테이터를 제공하는 `supervision` 라이브러리를 설치합니다. 설치 후에는 PyTorch가 코랩의 GPU(CUDA)를 정상적으로 인식하는지 확인합니다.


In [1]:
# RF-DETR 객체 검출 모델 및 Supervision(추적·시각화·에셋 다운로드) 패키지 설치
!pip install -q rfdetr "supervision>=0.25.0,<0.31.0" opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.9/788.9 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.7/409.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 5.5 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import argparse
import os
import subprocess
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import torch
import supervision as sv
from rfdetr import RFDETRSmall
from supervision.assets import VideoAssets, download_assets
from supervision.geometry.core import Position

# GPU(CUDA) 사용 가능 여부 확인 및 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("device:", device)

# GPU가 활성화된 경우 그래픽카드 모델명과 총 VRAM 용량(GB) 출력
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram_gb:", round(props.total_memory / 1024**3, 1))


torch: 2.11.0+cu130
device: cuda
gpu: Tesla T4
vram_gb: 14.6


## 2. 경로, 검출 대상 차종 및 예시 영상 명세(Specification) 설정

작업 디렉터리 경로(`ROOT`, `WORK`, `EXAMPLES_DIR`)를 지정하고, 전체 검출 클래스 중 집계할 4가지 차량 클래스(`car`, `truck`, `bus`, `motorcycle`)를 튜플로 정의합니다.

또한 `EXAMPLES` 딕셔너리에 각 예시 영상의 다운로드 출처, 기준선 방향(`horizontal`), 기준선의 세로 위치 비율(`line_y`), 가로 시작·끝 비율(`line_x0`, `line_x1`), 검출 신뢰도 임계값(`threshold`) 등 장면별 최적화된 파라미터를 설정합니다.


In [3]:
# 코랩 실행 환경에 맞춘 기본 작업 폴더 및 입출력 디렉터리 경로 설정
ROOT = Path("/content/line_count")
WORK = ROOT / "runs"              # 결과 동영상(.mp4)이 저장될 폴더
EXAMPLES_DIR = ROOT / "examples"  # 원본 예시 영상이 다운로드될 폴더

# 검출 및 카운팅 대상으로 삼을 4가지 차량 클래스 정의 (승용차, 트럭, 버스, 오토바이)
VEHICLE = ("car", "truck", "bus", "motorcycle")

# 강의 및 실습용 예시 영상 3종의 세부 설정 딕셔너리
EXAMPLES = {
    "vehicles": {
        "source": "supervision",      # Supervision 내장 에셋 사용
        "asset": "VEHICLES",
        "line": "horizontal",         # 가로 기준선 사용
        "line_y": 0.70,               # 화면 상단에서 70% 내려온 지점에 기준선 배치
        "threshold": 0.5,             # 객체 검출 최소 신뢰도 임계값
        "blurb": "기존 쿡북 고속도로.",
    },
    "a40": {
        "source": "url",              # 외부 웹 URL에서 직접 다운로드
        "url": "/" + "/".join(["https:", "", "commons.wikimedia.org", "wiki", "Special:FilePath", "Bochum_-_A40_(Br%C3%BCcke_L%C3%BCtkendorpweg)_01_(1)_ies.ogv"]).lstrip("/"),
        "filename": "a40.mp4",
        "convert_mp4": True,          # 원본 .ogv 영상을 ffmpeg로 .mp4 변환
        "line": "horizontal",
        "line_y": 0.58,
        "line_x0": 0.08,              # 기준선 왼쪽 끝 X 비율 (8%)
        "line_x1": 0.92,              # 기준선 오른쪽 끝 X 비율 (92%)
        "threshold": 0.35,
        "minimum_consecutive_frames": 3, # 최소 3프레임 연속 검출되어야 유효한 추적으로 인정
        "blurb": "보훔 A40. 고가에서 내려다본 양방향 아우토반.",
    },
    "a43": {
        "source": "url",
        "url": "/" + "/".join(["https:", "", "commons.wikimedia.org", "wiki", "Special:FilePath", "Bochum_-_A43_(Autobahnbr%C3%BCcke_In_der_Grume)_01_(1)_ies.ogv"]).lstrip("/"),
        "filename": "a43.mp4",
        "convert_mp4": True,
        "line": "horizontal",
        "line_y": 0.62,
        "line_x0": 0.04,
        "line_x1": 0.96,
        "threshold": 0.35,
        "blurb": "보훔 A43. 고가에서 내려다본 양방향 아우토반. 가까운 차가 크게 보인다.",
    },
}


## 3. 영상 다운로드 및 코덱 변환 함수 정의

선택한 예시 영상이 로컬 폴더(`examples/`)에 없으면 자동으로 내려받습니다.
- `vehicles` 예시는 `supervision.assets`를 통해 즉시 다운로드합니다.
- 위키미디어 공용(Wikimedia Commons)에서 가져오는 `a40`, `a43` 영상은 원본 포맷이 `.ogv`이므로, 다운로드 후 `ffmpeg`를 호출해 OpenCV 및 웹 브라우저 호환성이 높은 `H.264(.mp4)` 코덱으로 자동 변환합니다.


In [4]:
def fetch_url(url: str, filename: str, convert_mp4: bool = False) -> Path:
    """지정된 URL에서 영상 파일을 다운로드하고, 필요시 ffmpeg를 사용해 mp4(H.264) 포맷으로 변환하는 함수"""
    dest = EXAMPLES_DIR / filename
    EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)

    # 이미 1MB 이상의 정상 파일이 존재하면 재다운로드를 건너뛰고 경로 반환
    if dest.is_file() and dest.stat().st_size > 1_000_000:
        return dest.resolve()

    # 서버의 봇 차단 방지를 위한 User-Agent 및 Referer 헤더 설정
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
        "Referer": "/" + "/".join(["https:", "", "www.pexels.com", ""]).lstrip("/"),
    }
    print("받는 중:", url)

    # 변환이 필요한 경우 임시 확장자(.download)로 먼저 저장
    raw = dest if not convert_mp4 else dest.with_suffix(".download")
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=180) as src, open(raw, "wb") as out:
        out.write(src.read())

    # convert_mp4 옵션이 켜져 있으면 ffmpeg를 사용해 오디오를 제거(-an)하고 libx264 mp4로 인코딩
    if convert_mp4:
        subprocess.check_call(
            [
                "ffmpeg",
                "-y",
                "-i",
                str(raw),
                "-an",
                "-c:v",
                "libx264",
                "-pix_fmt",
                "yuv420p",
                str(dest),
            ]
        )
        # 변환 완료 후 임시 원본 파일 삭제
        raw.unlink(missing_ok=True)
    return dest.resolve()


def fetch_example(name: str) -> Path:
    """예시 이름(vehicles, a40, a43)에 따라 알맞은 방법으로 영상을 준비하고 파일 경로를 반환하는 함수"""
    spec = EXAMPLES[name]
    # 외부 URL 소스인 경우 fetch_url 호출
    if spec["source"] == "url":
        return fetch_url(spec["url"], spec["filename"], convert_mp4=bool(spec.get("convert_mp4")))

    # Supervision 내장 에셋인 경우 download_assets로 다운로드
    EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)
    asset = getattr(VideoAssets, spec["asset"])
    return Path(download_assets(asset, directory=EXAMPLES_DIR)).resolve()


def source_video(user: Path | None, example: str) -> Path:
    """사용자가 직접 지정한 영상 경로가 있으면 우선 사용하고, 없으면 예시 영상을 가져오는 함수"""
    if user is not None:
        path = user.expanduser().resolve()
        if not path.is_file():
            raise SystemExit(f"영상을 찾지 못했다: {path}")
        return path
    return fetch_example(example)


## 4. 기준선(LineZone) 생성 및 차량 검출 필터링

영상을 통과하는 차량을 세기 위해 **기준선(`sv.LineZone`)**과 **차량 검출 함수(`detect_vehicles`)**를 구현합니다.

- **기준선 방향과 IN/OUT 판정 원리**: `sv.LineZone`은 시작점(`start`)에서 끝점(`end`)을 바라보는 벡터를 기준으로 통과 방향을 판별합니다. 가로선(`horizontal`)을 만들 때 **시작점을 오른쪽(`x1`), 끝점을 왼쪽(`x0`)**으로 지정하면, 화면 아래로 다가오는 왼쪽 차로 차량이 **IN**, 화면 위로 멀어지는 오른쪽 차로 차량이 **OUT**으로 자연스럽게 분리되어 집계됩니다. 카운팅 기준점(`triggering_anchors`)은 차량 바운딩 박스의 **중심점(`Position.CENTER`)**만 사용합니다.
- **차량 클래스 및 최소 면적 필터링**: OpenCV의 BGR 프레임을 RGB로 변환한 뒤 `RFDETRSmall` 모델에 입력합니다. 검출된 결과 중 클래스 이름이 `VEHICLE`(`car`, `truck`, `bus`, `motorcycle`)에 해당하는 것만 남기고, 너무 작은 노이즈 박스를 걸러내기 위해 최소 면적(`min_area`) 필터를 적용합니다.


In [5]:
def crop_frame(frame, crop):
    """지정된 비율(x0, y0, x1, y1)에 따라 프레임의 특정 영역만 잘라내는(Crop) 함수"""
    if not crop:
        return frame
    h, w = frame.shape[:2]
    x0, y0, x1, y1 = crop
    return frame[int(h * y0) : int(h * y1), int(w * x0) : int(w * x1)]


def make_line_zone(sv, Position, spec: dict, width: int, height: int):
    """
    차량 통과를 감지할 기준선(LineZone) 객체를 생성하는 함수.
    차량 바운딩 박스의 중심(CENTER)이 선을 넘을 때만 카운트하며,
    가로선은 오른쪽(x1)에서 왼쪽(x0) 방향으로 설정하여 왼쪽 차로가 IN, 오른쪽 차로가 OUT이 되도록 구성한다.
    """
    anchors = (Position.CENTER,)
    # 세로 기준선(vertical)인 경우
    if spec.get("line") == "vertical":
        x = int(width * spec["line_x"])
        y0 = int(height * spec["line_y0"])
        y1 = int(height * spec["line_y1"])
        print("선 x =", x, "y =", y0, "~", y1)
        return sv.LineZone(
            start=sv.Point(x, y0),
            end=sv.Point(x, y1),
            triggering_anchors=anchors,
        )

    # 가로 기준선(horizontal)인 경우: 오른쪽(x1) -> 왼쪽(x0)으로 점 설정
    y = int(height * spec["line_y"])
    x0 = int(width * spec.get("line_x0", 0.0))
    x1 = int(width * spec.get("line_x1", 1.0))
    print("선 y =", y, "x =", x1, "->", x0)
    return sv.LineZone(
        start=sv.Point(x1, y),
        end=sv.Point(x0, y),
        triggering_anchors=anchors,
    )


def detect_vehicles(model, frame, threshold: float = 0.5, min_area: float = 0.0):
    """프레임에서 객체를 검출한 뒤, 차량 클래스(VEHICLE)와 최소 박스 면적 조건을 만족하는 결과만 필터링하는 함수"""
    # OpenCV BGR 이미지를 모델 입력용 RGB 연속 배열(Contiguous Array)로 변환
    rgb = np.ascontiguousarray(frame[:, :, ::-1])
    detections = model.predict(rgb, include_source_image=False, threshold=threshold)
    if len(detections) == 0:
        return detections

    # 검출된 객체 중 'car', 'truck', 'bus', 'motorcycle'에 해당하는 객체만 유지
    names = np.array(detections["class_name"])
    keep = np.isin(names, VEHICLE)
    detections = detections[keep]

    # 최소 면적(min_area) 기준이 설정된 경우 너무 작은 바운딩 박스 제거
    if min_area <= 0 or len(detections) == 0:
        return detections
    xyxy = detections.xyxy
    area = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    return detections[area >= min_area]


## 5. Tracker 역할 강조: 체류 시간(Dwell Time) 라벨링 및 비교 대시보드 배너

Object Tracker(`ByteTrack`)가 수행하는 핵심 역할을 시각적으로 명확히 보여주기 위해 두 가지 기능을 추가하여 화면을 구성합니다:

1. **차량별 화면 체류 시간(Dwell Time) 실시간 표시**:
   - 각 `tracker_id`가 최초로 등장한 프레임 인덱스를 `first_seen_frame` 딕셔너리에 기록합니다.
   - 현재 프레임과의 차이를 영상의 `fps`로 나누어 차량이 화면에 머문 시간($\Delta t = \frac{\text{현재 프레임} - \text{최초 프레임}}{\text{FPS}}$)을 계산하고, 바운딩 박스 라벨에 **`#4 car 0.85 (2.4s)`** 형태로 함께 표시합니다.
2. **'Tracker 없음(단순 검출 누적)' vs 'Tracker 적용' 비교 현황판**:
   - 하단 배너 상단 줄에 **단순 검출 박스 누적 합계(`Raw Box Sum`)**, **추적된 고유 차량 ID 총합(`Unique Tracked IDs`)**, 그리고 **실제 기준선 통과 차량 합계(`Line Crossed`)**를 나란히 출력합니다.
   - 이를 통해 단순 검출만으로는 한 차량이 수십 번 중복 카운트되는 반면, 트래커를 결합하면 어떻게 실제 차량 대수로 정확히 수렴하는지 실시간 수치로 비교할 수 있습니다.


In [6]:
def class_counts(line_zone):
    """LineZone 객체에 누적된 클래스별 IN / OUT 통과 대수를 차종 이름 기준의 딕셔너리로 정리하여 반환하는 함수"""
    in_counts = {name: 0 for name in VEHICLE}
    out_counts = {name: 0 for name in VEHICLE}

    # 클래스 ID와 클래스 이름을 매핑
    names = {int(k): v for k, v in line_zone.class_id_to_name.items() if k is not None}

    # IN 방향 차종별 카운트 합산
    for class_id, n in line_zone.in_count_per_class.items():
        if class_id is None:
            continue
        name = names.get(int(class_id))
        if name in in_counts:
            in_counts[name] += n

    # OUT 방향 차종별 카운트 합산
    for class_id, n in line_zone.out_count_per_class.items():
        if class_id is None:
            continue
        name = names.get(int(class_id))
        if name in out_counts:
            out_counts[name] += n

    return in_counts, out_counts


def class_banner_height(width: int) -> int:
    """비교 통계 줄과 차종별 대수를 모두 표시하기 위해 영상 가로 해상도에 맞춰 하단 배너 높이(픽셀)를 결정하는 함수"""
    return 100 if width < 800 else 124


def draw_class_panel(frame, line_zone, raw_box_total: int, unique_id_total: int):
    """
    영상 하단 막대(배너)에
    1) 상단 바: [단순 검출 박스 누적(Raw Boxes)] vs [고유 추적 차량 수(Unique IDs)] vs [기준선 통과 합계(Crossed)]
    2) 하단 좌/우 패널: 방향별(왼쪽 IN, 오른쪽 OUT) 및 차종별 통과 대수
    를 그려서 원본 프레임 아래에 수직으로 이어 붙이는 함수.
    """
    in_counts, out_counts = class_counts(line_zone)
    total_crossed = sum(in_counts.values()) + sum(out_counts.values())
    height, width = frame.shape[:2]
    narrow = width < 800
    banner_h = class_banner_height(width)
    top_bar_h = 28 if narrow else 34

    # 배너 캔버스 생성
    banner = np.zeros((banner_h, width, 3), dtype=np.uint8)
    mid = width // 2

    # 상단 비교 통계 바 배경색 (어두운 남색/회색 톤) 및 하단 좌(IN)/우(OUT) 배경색 설정
    banner[:top_bar_h, :] = (35, 35, 45)
    banner[top_bar_h:, :mid] = (28, 42, 28)   # 왼쪽(IN) 배경색 (BGR)
    banner[top_bar_h:, mid:] = (48, 32, 24)   # 오른쪽(OUT) 배경색 (BGR)

    # 구분선 그리기
    cv2.line(banner, (0, top_bar_h), (width, top_bar_h), (110, 110, 110), 1)
    cv2.line(banner, (mid, top_bar_h + 6), (mid, banner_h - 6), (90, 90, 90), 2)

    # 해상도에 따른 폰트 크기 및 여백 설정
    font = cv2.FONT_HERSHEY_SIMPLEX
    top_scale = 0.42 if narrow else 0.56
    title_scale = 0.60 if narrow else 0.78
    body_scale = 0.42 if narrow else 0.56
    title_th = 2
    body_th = 1 if narrow else 2
    pad = 12 if narrow else 20

    top_y = 19 if narrow else 23
    title_y = top_bar_h + (24 if narrow else 28)
    row0 = top_bar_h + (46 if narrow else 56)
    row1 = top_bar_h + (64 if narrow else 80)
    col_w = max(90, (mid - pad * 2) // 2)

    def put(text, x, y, scale, color, thickness):
        """배너 위에 안티앨리어싱이 적용된 텍스트를 출력하는 내부 헬퍼 함수"""
        cv2.putText(banner, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)

    # 1. 상단 비교 바: 단순 검출 누적 박스 수 vs Tracker 고유 ID 수 vs 기준선 통과 대수
    comparison_text = (
        f"[Tracker Effect]  Raw Box Sum (No Tracker): {raw_box_total}   |   "
        f"Unique Tracked IDs: {unique_id_total}   |   "
        f"Line Crossed (IN+OUT): {total_crossed}"
    )
    put(comparison_text, pad, top_y, top_scale, (120, 230, 255), body_th)

    # 2. 중단 대표 타이틀 (IN 총합 / OUT 총합) 출력
    put(f"IN  {sum(in_counts.values())}", pad, title_y, title_scale, (170, 255, 170), title_th)
    put(f"OUT  {sum(out_counts.values())}", mid + pad, title_y, title_scale, (190, 215, 255), title_th)

    # 3. 하단 4가지 차종(car, truck, bus, motorcycle)을 2열 2행 구조로 나란히 출력
    for i, name in enumerate(VEHICLE):
        col = i % 2
        y = row0 if i < 2 else row1
        put(
            f"{name}  {in_counts[name]}",
            pad + col * col_w,
            y,
            body_scale,
            (240, 240, 240),
            body_th,
        )
        put(
            f"{name}  {out_counts[name]}",
            mid + pad + col * col_w,
            y,
            body_scale,
            (240, 240, 240),
            body_th,
        )
    # 원본 프레임 하단에 배너를 수직으로 결합하여 반환
    return np.vstack([frame, banner])


def annotate(
    frame,
    detections,
    line_zone,
    box_annotator,
    label_annotator,
    trace_annotator,
    line_annotator,
    frame_index: int,
    fps: float,
    first_seen_frame: dict[int, int],
    raw_box_total: int,
):
    """
    프레임에 이동 궤적, 바운딩 박스, 체류 시간(초)이 포함된 라벨(#ID 차종 신뢰도 (체류초)), 기준선을 그리고
    하단에 Tracker 비교 집계 패널을 부착하는 함수.
    """
    names = detections["class_name"] if len(detections) else []
    if names is None:
        names = [""] * len(detections)

    # 각 차량의 추적 ID별 최초 등장 프레임을 기록하고, 현재 프레임까지의 체류 시간(초)을 계산하여 라벨 생성
    labels = []
    if len(detections):
        for tid, name, conf in zip(detections.tracker_id, names, detections.confidence):
            tid_int = int(tid)
            if tid_int not in first_seen_frame:
                first_seen_frame[tid_int] = frame_index
            # 체류 시간(Dwell Time) = (현재 프레임 번호 - 최초 등장 프레임 번호) / FPS
            dwell_sec = (frame_index - first_seen_frame[tid_int]) / fps
            labels.append(f"#{tid_int} {name} {conf:.2f} ({dwell_sec:.1f}s)")

    out = frame.copy()
    # 1. 차량 이동 궤적(꼬리선) 그리기
    out = trace_annotator.annotate(out, detections)
    # 2. 바운딩 박스 그리기
    out = box_annotator.annotate(out, detections)
    # 3. 체류 시간이 포함된 텍스트 라벨 그리기
    out = label_annotator.annotate(out, detections, labels)
    # 4. 기준선 통과 여부 판정(Trigger) 및 기준선 그리기
    line_zone.trigger(detections)
    out = line_annotator.annotate(out, line_counter=line_zone)
    # 5. 하단 비교 대시보드 및 차종별 대수 집계 패널 결합
    return draw_class_panel(out, line_zone, raw_box_total, len(first_seen_frame))


## 6. 비디오 처리 파이프라인 정의

입력 영상을 프레임 단위로 읽어들여 **크롭 및 리사이즈(1280×720) → RF-DETR 차량 검출 → ByteTrack 추적 ID 갱신 및 체류 시간 계산 → 기준선 카운팅 및 시각화 → MP4 파일 저장**을 수행하는 핵심 처리 루프(`process_video`)를 정의합니다.

- **누적 검출 수(`raw_box_total`) 및 체류 시간(`first_seen_frame`) 추적**: 매 프레임마다 검출된 박스 개수를 `raw_box_total`에 계속 더하고, `first_seen_frame` 딕셔너리로 각 고유 ID의 등장 시점을 관리합니다.
- **실시간 비교 로그 및 최종 리포트**: 25프레임마다 단순 검출 누적 개수와 고유 추적 ID 수, 실제 통과 대수를 비교 출력하며, 영상 처리가 끝나면 **단순 검출 대비 중복 제거율과 평균 체류 시간**을 요약해 보여줍니다.


In [7]:
def open_writer(path: Path, width: int, height: int, fps: float) -> cv2.VideoWriter:
    """결과 프레임을 MP4 동영상 파일로 저장하기 위한 OpenCV VideoWriter 객체를 생성하는 함수"""
    path.parent.mkdir(parents=True, exist_ok=True)
    fps = fps if fps and fps > 1 else 25.0
    writer = cv2.VideoWriter(
        str(path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        raise SystemExit(f"영상을 저장할 수 없다: {path}")
    return writer


def process_video(args, video_path: Path, out_path: Path, spec: dict, model, sv, Position) -> None:
    """단일 비디오 파일을 처음부터 끝까지(또는 max_frames까지) 읽으며 차량 검출·추적·체류시간 계산·카운팅 및 저장을 수행하는 메인 루프 함수"""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise SystemExit(f"영상을 열지 못했다: {video_path}")

    # 원본 영상의 FPS 확인 (인식 실패 시 기본값 25.0 사용)
    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = fps if fps and fps > 1 else 25.0
    width = int(spec.get("width", args.width))
    height = int(spec.get("height", args.height))
    crop = spec.get("crop")

    # 다중 객체 추적기(ByteTrack) 초기화
    tracker = sv.ByteTrack(
        frame_rate=fps,
        lost_track_buffer=int(spec.get("lost_track_buffer", 60)),
        minimum_matching_threshold=float(spec.get("minimum_matching_threshold", 0.5)),
        minimum_consecutive_frames=int(spec.get("minimum_consecutive_frames", 1)),
        track_activation_threshold=float(spec.get("track_activation_threshold", 0.25)),
    )

    # Supervision 시각화 어노테이터(박스, 라벨, 이동 궤적, 기준선) 초기화
    box_annotator = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(text_thickness=1, text_scale=0.5)
    trace_annotator = sv.TraceAnnotator(thickness=2)
    line_annotator = sv.LineZoneAnnotator(
        thickness=3,
        text_thickness=1,
        text_scale=0.8,
        display_in_count=False,  # 선 위의 기본 카운트 텍스트는 숨김 (하단 배너에 상세 표시)
        display_out_count=False,
    )

    # 프레임 높이 + 하단 집계 배너 높이만큼 출력 비디오 해상도 설정
    writer = open_writer(out_path, width, height + class_banner_height(width), fps)

    # 코랩과 같이 DISPLAY 환경 변수가 없는 서버 환경에서는 cv2.imshow를 자동으로 비활성화
    show = args.show and bool(os.environ.get("DISPLAY"))
    if args.show and not show:
        print("DISPLAY가 없어 cv2.imshow는 건너뛰고 파일만 저장한다.")
    if show:
        print("창에서 q를 누르면 멈춘다.")

    line_zone = None
    index = 0

    # Tracker 효과를 정량 비교하기 위한 상태 변수 초기화
    raw_box_total = 0                  # Tracker 없이 매 프레임 검출된 박스를 단순 합산한 개수
    first_seen_frame: dict[int, int] = {} # 각 tracker_id가 처음 등장한 프레임 번호
    last_seen_frame: dict[int, int] = {}  # 각 tracker_id가 마지막으로 관찰된 프레임 번호

    print("입력 영상:", video_path)
    print("기준선 방향:", spec.get("line", "horizontal"), "| 검출 임계값(threshold):", spec["threshold"])

    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break

        # 프레임 크롭 및 지정된 추론 해상도(기본 1280x720)로 리사이즈
        frame = crop_frame(frame, crop)
        frame = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)

        # 첫 프레임에서 기준선(LineZone) 객체 생성
        if line_zone is None:
            line_zone = make_line_zone(sv, Position, spec, width, height)

        # 1. 차량 검출 및 단순 박스 누적 개수 합산
        detections = detect_vehicles(
            model,
            frame,
            threshold=spec["threshold"],
            min_area=float(spec.get("min_area", 0)),
        )
        raw_box_total += len(detections)

        # 2. ByteTrack으로 검출 결과에 고유 추적 ID 업데이트
        detections = tracker.update_with_detections(detections)
        if detections.tracker_id is None:
            detections.tracker_id = np.zeros(len(detections), dtype=int)

        # 각 추적 ID의 마지막 관찰 프레임 갱신 (평균 체류 시간 산출용)
        for tid in detections.tracker_id:
            last_seen_frame[int(tid)] = index

        # 3. 시각화 어노테이션(체류 시간 포함) 및 기준선 통과 집계
        drawn = annotate(
            frame,
            detections,
            line_zone,
            box_annotator,
            label_annotator,
            trace_annotator,
            line_annotator,
            frame_index=index,
            fps=fps,
            first_seen_frame=first_seen_frame,
            raw_box_total=raw_box_total,
        )
        # 4. 결과 프레임 저장
        writer.write(drawn)

        if show:
            cv2.imshow("line_count", drawn)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord("q"), 27):
                break

        # 25프레임 간격으로 단순 누적 박스 수 vs 고유 ID 수 vs 통과 대수 비교 출력
        if index % 25 == 0:
            in_counts, out_counts = class_counts(line_zone)
            print(
                f"frame {index:03d} | raw_boxes={raw_box_total:4d} | "
                f"unique_ids={len(first_seen_frame):3d} | in={in_counts} | out={out_counts}"
            )

        index += 1
        # 최대 처리 프레임 수(max_frames)가 지정된 경우 해당 프레임 도달 시 종료
        if args.max_frames and index >= args.max_frames:
            break

    # 자원 해제
    cap.release()
    writer.release()
    if show:
        cv2.destroyAllWindows()

    # 최종 집계 결과 및 Object Tracker 효과 요약 리포트 출력
    in_counts, out_counts = ({name: 0 for name in VEHICLE}, {name: 0 for name in VEHICLE})
    if line_zone is not None:
        in_counts, out_counts = class_counts(line_zone)
    total_crossed = sum(in_counts.values()) + sum(out_counts.values())
    unique_total = len(first_seen_frame)
    avg_dwell = (
        np.mean([(last_seen_frame[tid] - first_seen_frame[tid]) / fps for tid in first_seen_frame])
        if first_seen_frame
        else 0.0
    )

    print("\n================ [최종 분석 요약] ================")
    print(f"- 총 처리 프레임 수          : {index} frames (FPS: {fps:.1f})")
    print(f"- 단순 검출 박스 누적 합계   : {raw_box_total} 개 (Tracker 미적용 시 중복 카운트)")
    print(f"- ByteTrack 고유 차량 ID 수  : {unique_total} 대 (평균 화면 체류 시간: {avg_dwell:.2f}초)")
    print(f"- 기준선 통과 차량 총합      : {total_crossed} 대 (IN: {sum(in_counts.values())}대, OUT: {sum(out_counts.values())}대)")
    print(f"  * 최종 IN 차종별 집계      : {in_counts}")
    print(f"  * 최종 OUT 차종별 집계     : {out_counts}")
    print(f"- 결과 동영상 저장 경로      : {out_path}")
    print("==================================================\n")


def jobs_from_args(args):
    """실행 인자(args)를 바탕으로 처리할 작업 목록(이름, 입력 경로, 출력 경로, 설정 딕셔너리)을 생성하는 함수"""
    # 사용자가 직접 비디오 경로를 지정한 경우
    if args.video is not None:
        spec = {
            "line": "horizontal",
            "line_y": 0.70 if args.line_y is None else args.line_y,
            "threshold": 0.5,
        }
        out = args.out if args.out is not None else WORK / "custom.mp4"
        return [("custom", source_video(args.video, "vehicles"), out.expanduser().resolve(), spec)]

    # 예시 영상(단일 또는 'all')을 실행하는 경우
    names = list(EXAMPLES) if args.example == "all" else [args.example]
    jobs = []
    for name in names:
        spec = dict(EXAMPLES[name])
        if spec.get("line", "horizontal") == "horizontal" and args.line_y is not None:
            spec["line_y"] = args.line_y
        out = WORK / f"{name}.mp4" if args.example == "all" or args.out is None else args.out
        jobs.append((name, fetch_example(name), Path(out).expanduser().resolve(), spec))
    return jobs


## 7. 차량 카운팅 실행

코랩 노트북에서 바로 옵션을 바꿔가며 실험할 수 있도록 실행 파라미터(`EXAMPLE`, `MAX_FRAMES`, `LINE_Y` 등)를 설정하고 `RFDETRSmall` 모델을 로드하여 차량 카운팅을 실행합니다.

- `EXAMPLE`: `"vehicles"`, `"a40"`, `"a43"`, 또는 3가지를 모두 실행하는 `"all"` 중 하나를 선택할 수 있습니다.
- `MAX_FRAMES`: 빠른 테스트를 위해 기본값을 `150` 프레임으로 설정했습니다. 영상 끝까지 전체를 처리하려면 `0`으로 변경하세요.


In [8]:
# === 코랩 실습용 파라미터 설정 ===
EXAMPLE = "vehicles"  # 선택 가능: "vehicles", "a40", "a43", "all"
VIDEO_PATH = None     # 직접 업로드한 영상을 쓰려면 Path("/content/my_video.mp4") 입력
WIDTH = 1280          # 추론 및 출력 영상 가로 해상도
HEIGHT = 720          # 추론 및 출력 영상 세로 해상도
LINE_Y = None         # 기준선 세로 위치 비율 직접 지정 (None이면 예시별 기본값 사용)
MAX_FRAMES = 0      # 테스트용 최대 처리 프레임 수 (0으로 설정하면 영상 끝까지 처리)
OUT_PATH = None       # 출력 파일 경로 (None이면 /content/line_count/runs/{EXAMPLE}.mp4 에 자동 저장)

# 기존 스크립트의 argparse.Namespace와 동일한 구조로 인자 객체 생성
args = argparse.Namespace(
    example=EXAMPLE,
    video=VIDEO_PATH,
    width=WIDTH,
    height=HEIGHT,
    line_y=LINE_Y,
    max_frames=MAX_FRAMES,
    out=OUT_PATH,
    show=False,
)

# RF-DETR Small 모델 로드 및 비디오 처리 작업 실행
model = RFDETRSmall()
jobs = jobs_from_args(args)

for name, video_path, out_path, spec in jobs:
    if name in EXAMPLES:
        print("\n====================================")
        print("예시 실행:", name, "-", EXAMPLES[name]["blurb"])
        print("====================================")
    process_video(args, video_path, out_path, spec, model, sv, Position)


[2026-10-18 18:53:55] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-small.pth


/root/.roboflow/models/rf-detr-small.pth:   0%|          | 0.00/368M [00:00<?, ?iB/s]

[2026-10-18 18:54:17] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-small.pth


[2026-10-18 18:54:17] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-18 18:54:17] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-10-18 18:54:18] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.
[2026-10-18 18:54:20] [INFO] supervision.assets.downloader - Downloading vehicles.mp4 assets


...


예시 실행: vehicles - 기존 쿡북 고속도로.
입력 영상: /content/line_count/examples/vehicles.mp4
기준선 방향: horizontal | 검출 임계값(threshold): 0.5


/tmp/ipykernel_2421/763555926.py:30: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  tracker = sv.ByteTrack(


선 y = 503 x = 1280 -> 0


[2026-10-18 18:54:26] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).


frame 000 | raw_boxes=   4 | unique_ids=  4 | in={'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0} | out={'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0}
frame 025 | raw_boxes= 135 | unique_ids= 11 | in={'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0} | out={'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0}
frame 050 | raw_boxes= 304 | unique_ids= 15 | in={'car': 2, 'truck': 1, 'bus': 0, 'motorcycle': 0} | out={'car': 0, 'truck': 0, 'bus': 0, 'motorcycle': 0}
frame 075 | raw_boxes= 460 | unique_ids= 18 | in={'car': 2, 'truck': 1, 'bus': 0, 'motorcycle': 0} | out={'car': 1, 'truck': 0, 'bus': 0, 'motorcycle': 0}
frame 100 | raw_boxes= 594 | unique_ids= 20 | in={'car': 2, 'truck': 1, 'bus': 0, 'motorcycle': 0} | out={'car': 1, 'truck': 0, 'bus': 0, 'motorcycle': 0}
frame 125 | raw_boxes= 719 | unique_ids= 20 | in={'car': 2, 'truck': 1, 'bus': 0, 'motorcycle': 0} | out={'car': 1, 'truck': 0, 'bus': 0, 'motorcycle': 0}
frame 150 | raw_boxes= 844 | unique_ids= 20 | in={'car': 2, 'truck': 1

## 8. 결과 동영상 코랩 화면에서 재생하기

OpenCV의 `mp4v` 코덱으로 생성된 결과 영상을 웹 브라우저(코랩 출력창)에서 바로 재생할 수 있도록 `ffmpeg`를 이용해 `H.264` 웹 표준 포맷으로 재인코딩한 뒤 화면에 띄웁니다. 각 차량 라벨의 **체류 시간(`s`)**과 하단 배너 상단의 **`[Tracker Effect]` 수치 비교**를 확인해 보세요.


In [9]:
from IPython.display import HTML, display
from base64 import b64encode

def show_video_in_colab(video_path: Path, display_width: int = 800):
    """OpenCV로 저장된 mp4 파일을 H.264로 변환하여 코랩 브라우저 출력창에 비디오 플레이어로 띄우는 함수"""
    if not video_path.is_file():
        print(f"파일을 찾을 수 없습니다: {video_path}")
        return

    h264_path = video_path.with_name(f"{video_path.stem}_web.mp4")
    # 브라우저 호환을 위해 libx264 코덱으로 빠르게 변환
    subprocess.check_call([
        "ffmpeg", "-y", "-i", str(video_path),
        "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-an",
        "-loglevel", "quiet", str(h264_path)
    ])

    mp4 = open(h264_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width="{display_width}" controls autoplay loop muted>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))

# 방금 처리한 첫 번째 작업의 결과 영상 재생
if jobs:
    _, _, last_out_path, _ = jobs[0]
    print("결과 영상 재생:", last_out_path)
    show_video_in_colab(last_out_path)


결과 영상 재생: /content/line_count/runs/vehicles.mp4
